# Full Day Systematics Inspection

**by Tyler Cox and Josh Dillon**, last updated November 3, 2026

This notebook inspects the night's final product, the inpainted single-baseline data written by
[single_baseline_wiener_inpaint](https://github.com/HERA-Team/hera_notebook_templates/blob/master/notebooks/phase_II/single_baseline_wiener_inpaint.ipynb),
in delay and fringe rate on a few redundantly-averaged baselines. It is the last step of the per-night pipeline (was H6C's
[post_corner_turn_systematics_inspect](https://github.com/HERA-Team/hera_notebook_templates/blob/master/notebooks/post_corner_turn_systematics_inspect.ipynb),
which was adapted from
[a similar notebook written by Josh Dillon for H1C analysis](https://github.com/HERA-Team/hera_notebook_templates/blob/master/notebooks/h1c_systematics_mitigation_inspect.ipynb)).

It looks at:
* the 1-, 2- and 4-unit (14.6, 29.2 and 58.4 m) east-west baselines, and one inter-sector baseline from antenna 144;
* both linear polarizations, then pseudo-Stokes I and Q;
* two wide bands, one below FM and one above.

For each, it shows delay spectra against LST, delay spectra against fringe rate, and delay spectra averaged over the night,
at four stages:
* **Section 1:** the inpainted data. What the inpainting left flagged (integrations flagged across a whole band) is filled
  in here too, by a DPSS fit in time.
* **Section 2:** after a delay filter (out to the horizon plus `STANDOFF`, and at least `FILT_MIN_DLY`) and a cross-talk
  filter (a notch at zero fringe rate, of half-width `XTALK_FRATE`, as in the pI SNR stage).
* **Section 3:** the same, as pseudo-Stokes I and Q.
* **Section 4:** after a filter that keeps only the main beam's fringe rates, from a beam model.

The polarized point source models were subtracted before inpainting, so the data here are source-subtracted.

Configuration comes from a TOML file (e.g.
`hera_pipelines/pipelines/phase_II/idr1/v1/analysis/phase_II_analysis.toml`) pointed to by the
`TOML_FILE` environment variable; if none is given, the default settings in the cells below are
used. Environment variables otherwise carry only paths.

Here's a set of links to skip to particular sections:

• [Section 1: Inpainted Redundantly-Averaged Calibrated Visibilities](#Section-1:-Inpainted-Redundantly-Averaged-Calibrated-Visibilities)

• [Section 2: Delay-Filter and Cross-Talk Filter Redundantly-Averaged Visibilities](#Section-2:-Delay-Filter-and-Cross-Talk-Filter-Redundantly-Averaged-Visibilities)

• [Section 3: Form Pseudo-Stokes I + Q](#Section-3:-Form-Pseudo-Stokes-I-+-Q)

• [Section 4: Main-Beam Fringe-Rate Filter](#Section-4:-Main-Beam-Fringe-Rate-Filter)

In [ ]:
import time
tstart = time.time()
!hostname
!date

In [ ]:
import os
import toml
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
import h5py
import hdf5plugin  # REQUIRED to have the compression plugins available
import re
import yaml
import uvtools
import matplotlib
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy
from pyuvdata import UVBeam
from hera_filters import dspec
from hera_cal import io, frf, vis_clean
from hera_cal.red_groups import RedundantGroups
from hera_cal.datacontainer import DataContainer, RedDataContainer
from IPython.display import display, HTML

%matplotlib inline
display(HTML("<style>.container { width:100% !important; }</style>"))
_ = np.seterr(all='ignore')  # get rid of red warnings
%config InlineBackend.figure_format = 'retina'

## Parse inputs

To run interactively, set `SUM_FILE` (and `TOML_FILE`, if wanted) in the next cell; everything else has defaults. To reproduce a
run instead, paste the settings that cell printed in the run's HTML into the cell after it. Everything derived from the
settings is computed after that.

In [ ]:
# parse wrapper-level environment variables: paths
SUM_FILE = os.environ.get("SUM_FILE", None)
# SUM_FILE = '/lustre/aoc/projects/hera/phase-II-analysis/idr1/2459935/zen.2459935.21341.sum.uvh5'
TOML_FILE = os.environ.get("TOML_FILE", None)
# TOML_FILE = '/lustre/aoc/projects/hera/phase-II-analysis/idr1/src/hera_pipelines/pipelines/phase_II/idr1/v1/analysis/phase_II_analysis.toml'
# which [WorkFlow] action is running this notebook
ACTION = os.environ.get("ACTION", "FULL_DAY_SYSTEMATICS_INSPECT_NOTEBOOK")

# default names for the files this notebook reads: single-baseline files end in the suffix, and per-night products are
# relative to SUM_FILE's folder
INPAINTED_SUFFIX = 'sum.inpainted.uvh5'
CORNER_TURN_MAP_FILENAME = 'single_baseline_files/corner_turn_map.yaml'

# shared across the pipeline via [GLOBAL_OPTS]
BAND_SPLIT_FREQ = 100.0  # in MHz; divides the low and high bands (inside the a priori flagged FM gap)

# the pI SNR stage's, from [SINGLE_BASELINE_PI_SNR_OPTS], so that the cross-talk-filtered views here match what its flagging rounds used
XTALK_FRATE = 0.01  # in mHz; half-width of the cross-talk filter's notch at zero fringe rate

# default settings, overridden by the TOML's [FULL_DAY_SYSTEMATICS_INSPECT_OPTS] section (if given)
FILT_MIN_DLY = 150.0  # in ns; the delay filter reaches the horizon plus STANDOFF, and at least this far
STANDOFF = 50.0  # in ns
INPAINT_FR = 2.5  # in mHz; what the inpainting left flagged (whole integrations) is filled by a DPSS fit in time out to this fringe rate...
INPAINT_REGULARIZATION = 1e-5  # ...with this ridge; reasonable values are between 1e-2 and 1e-5
SPECTRUM_FREQ_BUFFER = 3.0  # in MHz; left out of the delay spectra at both ends of each band
EIGENVAL_CUTOFF = 1e-12
UVBEAM_FILE = '/lustre/aoc/projects/hera/phase-II-analysis/idr1/beam_simulation_products/NF_HERA_Vivaldi_efield_beam_healpix.fits'  # the beam model that sets the main beam's fringe rates

toml_options = (toml.load(TOML_FILE) if TOML_FILE is not None else {})

# Take the suffixes this notebook reads or writes from [DATA_PRODUCTS], scoped by that
# section's produced_by/consumed_by wiring, and require that wiring to agree exactly with
# the defaults above. That way the arrows drawn on the pipeline flowchart cannot drift from
# what this notebook actually touches: adding an edge there without using the file here (or
# vice versa) fails loudly, in the first cell, rather than silently.
declared_suffixes = {name for name in list(globals()) if name.endswith('_SUFFIX')}
wired_suffixes = set()
for product, spec in toml_options.get('DATA_PRODUCTS', {}).items():
    producers, consumers = (spec.get(key, []) for key in ['produced_by', 'consumed_by'])
    producers = ([producers] if isinstance(producers, str) else producers)
    consumers = ([consumers] if isinstance(consumers, str) else consumers)
    if 'suffix' in spec and (ACTION in producers or ACTION in consumers):
        wired_suffixes.add(f'{product.upper()}_SUFFIX')
        globals()[f'{product.upper()}_SUFFIX'] = spec['suffix']
if toml_options:
    assert wired_suffixes == declared_suffixes, (
        f'[DATA_PRODUCTS] wires {sorted(wired_suffixes)} to {ACTION}, '
        f'but this notebook declares {sorted(declared_suffixes)}.')
# per-night products have no per-file suffix; their [DATA_PRODUCTS] filenames are relative to SUM_FILE's folder
for product in [name[:-len('_FILENAME')] for name in list(globals()) if name.endswith('_FILENAME')]:
    globals()[f'{product}_FILENAME'] = toml_options.get('DATA_PRODUCTS', {}).get(product, {}).get('filename', globals()[f'{product}_FILENAME'])

for toml_section in ['GLOBAL_OPTS', 'FULL_DAY_SYSTEMATICS_INSPECT_OPTS']:
    if toml_section in toml_options:
        print(f'Loading overrides from [{toml_section}] in {TOML_FILE}.')
        for key, val in toml_options[toml_section].items():
            globals()[key.upper()] = val
if 'xtalk_frate' in toml_options.get('SINGLE_BASELINE_PI_SNR_OPTS', {}):
    print(f'Loading XTALK_FRATE from [SINGLE_BASELINE_PI_SNR_OPTS] in {TOML_FILE}.')
    XTALK_FRATE = toml_options['SINGLE_BASELINE_PI_SNR_OPTS']['xtalk_frate']

def settings_note():
    '''The settings above as Python assignments. Pasted into a cell right after this one, the printed note reproduces a run
    interactively, since everything derived from the settings is computed after it.'''
    return '\n'.join(f'{setting} = {eval(setting)!r}' for setting in [
        'SUM_FILE', 'TOML_FILE', 'ACTION', 'INPAINTED_SUFFIX', 'CORNER_TURN_MAP_FILENAME', 'BAND_SPLIT_FREQ', 'FILT_MIN_DLY',
        'STANDOFF', 'XTALK_FRATE', 'INPAINT_FR', 'INPAINT_REGULARIZATION', 'SPECTRUM_FREQ_BUFFER',
        'EIGENVAL_CUTOFF', 'UVBEAM_FILE'])

print(settings_note())

In [ ]:
# copy-paste settings note here to interactively override the above defaults or whatever comes from the TOML

In [ ]:
# everything derived from the settings, computed after them so that settings pasted in to reproduce a run carry through
jdstr = re.search(r'zen\.(\d+)\.', os.path.basename(SUM_FILE)).group(1)
CORNER_TURN_MAP_YAML = os.path.join(os.path.dirname(SUM_FILE), CORNER_TURN_MAP_FILENAME)
print(f'CORNER_TURN_MAP_YAML = {CORNER_TURN_MAP_YAML}')

## Load data

In [ ]:
# the night's redundant groups, by the keys that name their single-baseline files
with open(CORNER_TURN_MAP_YAML, 'r') as file:
    corner_turn_map = yaml.unsafe_load(file)
ubl_keys = sorted({tuple(int(ant) for ant in key) for key in corner_turn_map['antpairs_to_ubl_keys_map'].values()})
print(f'Found {len(ubl_keys)} redundant groups in the corner-turn map.')

def inpainted_file(ubl_key):
    '''The night's inpainted file for the redundant group with this key: whole-night single-baseline files are named
    zen.{JD}.baseline.{ant1}_{ant2}.{suffix} for their group's key, in the corner-turn map's folder.'''
    return os.path.join(os.path.dirname(CORNER_TURN_MAP_YAML), f'zen.{jdstr}.baseline.{ubl_key[0]}_{ubl_key[1]}.{INPAINTED_SUFFIX}')

# the positions of every antenna, not only those with data
for key in ubl_keys:
    if os.path.exists(inpainted_file(key)):
        antpos = io.HERAData(inpainted_file(key)).antpos
        break
else:
    raise FileNotFoundError(f'None of the {len(ubl_keys)} inpainted files in {os.path.dirname(CORNER_TURN_MAP_YAML)} exist.')

In [ ]:
# the 1-, 2- and 4-unit east-west baselines...
bls_to_plot = []
for n_units in [1, 2, 4]:
    for key in ubl_keys:
        bl_vec = antpos[key[1]] - antpos[key[0]]
        if (np.abs(bl_vec[1]) < 1) and (int(np.round(np.abs(bl_vec[0]) / 14.6)) == n_units):
            if os.path.exists(inpainted_file(key)):
                bls_to_plot += [key + ('ee',), key + ('nn',)]
            else:
                print(f'The {n_units}-unit east-west baseline {key} has no inpainted file (likely skipped upstream).')
            break

# ...and the first of these inter-sector baselines whose median nsamples are positive in both polarizations
reds = RedundantGroups.from_antpos(antpos, include_autos=False).keyed_on_bls(ubl_keys)
for ant2 in list(range(226, 233)) + list(range(207, 215)) + list(range(187, 196)):
    try:
        key = tuple(int(ant) for ant in reds.get_ubl_key((144, ant2)))
    except KeyError:
        continue
    if (key not in ubl_keys) or not os.path.exists(inpainted_file(key)):
        continue
    _, _, nsamples_test = io.HERAData(inpainted_file(key)).read(polarizations=['ee', 'nn'])
    if all(np.median(nsamples_test[key + (pol,)]) > 0 for pol in ['ee', 'nn']):
        bls_to_plot += [key + ('ee',), key + ('nn',)]
        break

if len(bls_to_plot) < 8:
    print(f'Warning: only found {len(bls_to_plot)} baselines for plotting (expected 8).')
print(f'Selected {len(bls_to_plot)} baselines for plotting: {bls_to_plot}')

In [ ]:
ubl_keys_to_plot = list(dict.fromkeys(bl[:2] for bl in bls_to_plot))
print(f'Loading {len(ubl_keys_to_plot)} inpainted files.')
hd = io.HERAData([inpainted_file(key) for key in ubl_keys_to_plot])
d, f, n = hd.read(polarizations=['nn', 'ee'])

In [ ]:
data = RedDataContainer(deepcopy(d), antpos=d.antpos)
flags = RedDataContainer(deepcopy(f), antpos=f.antpos)
nsamples = RedDataContainer(deepcopy(n), antpos=n.antpos)
antpos = data.antpos

# figure out bands: below and above BAND_SPLIT_FREQ, each from its first channel with any unflagged data to its last
split_ind = np.argmin(np.abs(data.freqs - BAND_SPLIT_FREQ * 1e6))
unflagged_chans = np.argwhere(~np.all([np.all(flags[bl], axis=0) for bl in bls_to_plot], axis=0)).squeeze()
low_band = slice(np.min(unflagged_chans), np.max(unflagged_chans[unflagged_chans < split_ind]) + 1)
high_band = slice(np.min(unflagged_chans[unflagged_chans > split_ind]), np.max(unflagged_chans) + 1)

# Create a truncated version of the data to deal with edge times that are flagged
unflagged_times = np.where((~np.all(flags[bls_to_plot[0]][:, low_band], axis=1)) &
                           (~np.all(flags[bls_to_plot[0]][:, high_band], axis=1)))[0]
inds = np.arange(np.min(unflagged_times), np.max(unflagged_times) + 1)

# Fix data
for bl in data:
    data[bl] = d[bl][inds]
    flags[bl] = f[bl][inds]
    nsamples[bl] = n[bl][inds]

# Fix metadata
data.times = d.times[inds]
data.lsts = d.lsts[inds]

# *Section 1: Inpainted Redundantly-Averaged Calibrated Visibilities*

Figure 1 shows the 1-unit baseline over the night, blank where the inpainting left it flagged: inside the bands, only
integrations flagged across a whole band. From Figure 2 on, those are filled in too, by a DPSS fit in time out to `INPAINT_FR`,
so that the fringe-rate transforms see no gaps. Then, for each baseline, below and above FM:
* Figures 2 and 3: the real part of each integration's delay spectrum, against LST, with the delay filter's edges (Section 2) marked;
* Figures 4 and 5: the amplitude of the delay and fringe-rate transform.

In [ ]:
unit = int(np.round(np.linalg.norm(antpos[bls_to_plot[0][1]] - antpos[bls_to_plot[0][0]]) / 14.6, 1))
display(HTML(f'<h2>Figure 1: Amplitude and Phase of the Inpainted, Calibrated {unit}-unit EW Redundantly-Averaged Baseline</h2>'))

# Get the LSTs from the data
lsts_unwrapped = np.where(data.lsts > data.lsts[-1], data.lsts - 2 * np.pi, data.lsts) * 12 / np.pi  # in hours
extent = [data.freqs[0] / 1e6, data.freqs[-1] / 1e6, lsts_unwrapped[-1], lsts_unwrapped[0]]

fig, axs = plt.subplots(2, 2, figsize=(18, 14), sharex=True, sharey=True, gridspec_kw={'hspace': .03, 'wspace': .03})
fig.subplots_adjust(hspace=0.1, wspace=0.1)

# Plot the amplitude of ee and nn
im1 = axs[0, 0].imshow(np.where(flags[bls_to_plot[0]], np.nan, np.abs(data[bls_to_plot[0]])), aspect='auto', interpolation='none',
                       norm=matplotlib.colors.LogNorm(), cmap='inferno', extent=extent)
axs[1, 0].imshow(np.where(flags[bls_to_plot[1]], np.nan, np.abs(data[bls_to_plot[1]])), aspect='auto', interpolation='none',
                 norm=matplotlib.colors.LogNorm(vmin=im1.get_clim()[0], vmax=im1.get_clim()[1]), cmap='inferno', extent=extent)

# Plot the phase of ee and nn
im2 = axs[0, 1].imshow(
    np.where(flags[bls_to_plot[0]], np.nan, np.angle(data[bls_to_plot[0]])),
    aspect='auto', cmap='twilight', interpolation='none', extent=extent,
    vmin=-np.pi, vmax=np.pi
)
axs[1, 1].imshow(
    np.where(flags[bls_to_plot[1]], np.nan, np.angle(data[bls_to_plot[1]])),
    aspect='auto', cmap='twilight', interpolation='none', extent=extent,
    vmin=-np.pi, vmax=np.pi
)

fig.colorbar(im1, ax=axs[:, 0], label='$|V_{ij}|$ (Jy)', location='top', pad=0.01)
fig.colorbar(im2, ax=axs[:, 1], label='Angle($V_{ij})$ (radians)', location='top', pad=0.01)

for ax, pol in zip(axs.ravel(), [bls_to_plot[0][2]] * 2 + [bls_to_plot[1][2]] * 2):
    ax.text(0.03, 0.97, f"{unit}-unit EW ({pol})",
             horizontalalignment='left', verticalalignment='top',
             color='black', transform=ax.transAxes, fontsize=12,
             bbox=dict(facecolor='w', alpha=.7, boxstyle='round'))

for i in range(axs.shape[1]):
    axs[1, i].set_xlabel('Frequency (MHz)')

for i in range(axs.shape[0]):
    axs[i, 0].set_ylabel('LST (hours)')

plt.tight_layout()

In [ ]:
def sym_log_norm(to_plot):
    '''Shortcut for symmetric sym log norm.'''
    return matplotlib.colors.SymLogNorm(10, vmin=-np.nanmax(np.abs(to_plot)), vmax=np.nanmax(np.abs(to_plot)))

In [ ]:
# Copy data
data_inpaint = deepcopy(data)
data_delay_filt = deepcopy(data)

# Fill in what the inpainting left flagged (inside the bands, whole integrations) by a DPSS fit in time, then delay filter,
# each separately below and above BAND_SPLIT_FREQ
delay_bounds = {}
cache = {}

for bl in data:
    if np.all(data[bl] == 0.0) and np.all(flags[bl]):
        # don't bother delay filtering all 0s.
        continue

    wgts = np.where(flags[bl], 0, 1)

    # inpaint in time
    d_mdl = np.zeros_like(data[bl])
    for band in [low_band, high_band]:
        d_mdl[:, band], _, info = dspec.fourier_filter(data.times * 24 * 60 * 60, data[bl][:, band], wgts=wgts[:, band], filter_centers=[0],
                                                       filter_half_widths=[INPAINT_FR / 1000.], mode='dpss_solve', filter_dims=0, ridge_alpha=INPAINT_REGULARIZATION,
                                                       eigenval_cutoff=[EIGENVAL_CUTOFF], max_contiguous_edge_flags=len(data.times), cache=cache)
    data_inpaint[bl] = np.where(flags[bl], d_mdl, data[bl])

    # calculate filter properties
    bl_vec = (data.antpos[bl[1]] - data.antpos[bl[0]])
    bl_len = np.linalg.norm(bl_vec[:2]) / vis_clean.constants.c.value
    filter_centers, filter_half_widths = vis_clean.gen_filter_properties(ax='freq', horizon=1, standoff=STANDOFF,
                                                                         min_dly=FILT_MIN_DLY, bl_len=bl_len)
    delay_bounds[bl[:2]] = (filter_centers[0] - filter_half_widths[0], filter_centers[0] + filter_half_widths[0])

    # run delay filter for each band individually
    d_mdl = np.zeros_like(data[bl])
    for band in [low_band, high_band]:
        d_mdl[:, band], _, info = dspec.fourier_filter(data.freqs[band], data[bl][:, band], wgts=wgts[:, band], filter_centers=filter_centers,
                                                       filter_half_widths=filter_half_widths, mode='dpss_solve',
                                                       eigenval_cutoff=[EIGENVAL_CUTOFF], max_contiguous_edge_flags=len(data.freqs[band]), cache=cache)
    data_delay_filt[bl] = np.where(flags[bl], 0, data[bl] - d_mdl)

In [ ]:
# Indices to use for plotting: each band but SPECTRUM_FREQ_BUFFER at both ends
n_buffer = int(np.round(SPECTRUM_FREQ_BUFFER * 1e6 / np.median(np.diff(data.freqs))))
low_rng = np.arange(low_band.start + n_buffer, low_band.stop - n_buffer, dtype=int)
high_rng = np.arange(high_band.start + n_buffer, high_band.stop - n_buffer, dtype=int)

In [ ]:
def plot_delay_vs_lst(data, rng, min_dly=FILT_MIN_DLY, vmin=None, vmax=None, title=""):
    
    # Add search-able title to figure
    display(HTML(f'<h2>{title}</h2>'))
    
    # plot in delay space
    fig, axs = plt.subplots(2, 4, figsize=(14,14), sharex=True, sharey=True)

    # Organize baselines
    plot_bls = list(set([bl[:2] for bl in data.keys() if bl[0] != bl[1]]))
    plot_bls = sorted(plot_bls, key=lambda k: np.linalg.norm(antpos[k[1]] - antpos[k[0]]))
    plot_pols = sorted(list(set([bl[-1] for bl in data.keys()])))

    # Fringe-rates
    lsts_unwrapped = np.where(data.lsts > data.lsts[-1], data.lsts - 2 * np.pi, data.lsts) * 12 / np.pi  # in hours

    for bi, blkey in enumerate(plot_bls):
        for pi, pol in enumerate(plot_pols):
            bl = blkey + (pol, )
            dfft = uvtools.utils.FFT(data[bl][:, rng], axis=1, taper='bh7')
            delays = uvtools.utils.fourier_freqs(data.freqs[rng]) * 1e9
            to_plot = np.real(dfft)

            # Scale vmin and vmax 
            if bi == 0 and pi == 0: 
                _to_plot = np.copy(to_plot)

            im = axs[pi, bi].imshow(to_plot, interpolation='none', aspect='auto', cmap='bwr', norm=sym_log_norm(_to_plot),
                       extent=[delays[0], delays[-1], lsts_unwrapped[-1], lsts_unwrapped[0]])
            axs[pi, bi].set_xlim([-1499, 1499])

            # Get baseline-lengths
            blvec = antpos[bl[0]] - antpos[bl[1]]
            bllen = np.linalg.norm(blvec)
            if bi < 3:
                bltype = f"{int(np.round(bllen / 14.6))}-unit EW" 
            else:
                bltype = f"{np.abs(blvec[0]):.0f} m {'E' if blvec[0] > 0 else 'W'}, {np.abs(blvec[1]):.0f} m {'N' if blvec[1] > 0 else 'S'}"

            axs[pi, bi].text(0.05, 0.98, f"{bltype} ({pol})\n{np.round(data.freqs[rng[0]] / 1e6, 1)}—{np.round(data.freqs[rng[-1]] / 1e6, 1)} MHz", 
                             horizontalalignment='left', verticalalignment='top', 
                             color='black', transform=axs[pi, bi].transAxes, fontsize=12, 
                             bbox=dict(facecolor='w', alpha=.7, boxstyle='round'))

            # Label axes
            if pi == 1:
                axs[pi, bi].set_xlabel('Delay (ns)')
            if bi == 0:
                axs[pi, bi].set_ylabel('LST [hr]')
                
            bl_vec = (antpos[blkey[1]] - antpos[blkey[0]])
            bl_len = np.linalg.norm(bl_vec[:2]) / vis_clean.constants.c.value
            filter_centers, filter_half_widths = vis_clean.gen_filter_properties(
                ax='freq', horizon=1, standoff=STANDOFF, min_dly=min_dly, bl_len=bl_len
            )

            axs[pi, bi].axvline(-filter_half_widths[0] * 1e9, color='k', ls='-.')
            axs[pi, bi].axvline(filter_half_widths[0] * 1e9, color='k', ls='-.')
    
    plt.tight_layout()

    # Colorbar
    fig.subplots_adjust(right=0.9)
    cbar_ax = fig.add_axes([0.92, 0.04, 0.02, 0.95])
    fig.colorbar(im, cax=cbar_ax, label=r'$\mathrm{Re}(\widetilde{V})\,\mathrm{[Jy]}$', extend='both')
    return {'vmin': vmin, 'vmax': vmax}

In [ ]:
def plot_delay_fringe_rate(data, rng, vmin=None, vmax=None, delay_bounds=None, fringe_bounds=None, title=""):
    
    # Add search-able title to figure
    display(HTML(f'<h2>{title}</h2>'))
    
    # plot in delay space
    fig, axs = plt.subplots(2, 4, figsize=(14, 8), sharex=True, sharey=True)

    # Organize baselines
    plot_bls = list(set([bl[:2] for bl in data.keys() if bl[0] != bl[1]]))
    plot_bls = sorted(plot_bls, key=lambda k: np.linalg.norm(antpos[k[1]] - antpos[k[0]]))
    plot_pols = sorted(list(set([bl[-1] for bl in data.keys()])))

    # Fringe-rates
    frates = uvtools.utils.fourier_freqs(data.times * 24 * 60 * 60) * 1000

    for bi, blkey in enumerate(plot_bls):
        for pi, pol in enumerate(plot_pols):
            bl = blkey + (pol, )
            dfft = uvtools.utils.FFT(data[bl][:, rng], axis=1, taper='bh7')
            delays = uvtools.utils.fourier_freqs(data.freqs[rng]) * 1e9
            dfft2 = uvtools.utils.FFT(dfft, axis=0, taper='bh7')

            # Scale vmin and vmax 
            if bi == 0 and pi == 0: 
                vmin = np.median(np.abs(dfft2)) if vmin is None else vmin
                vmax = np.max(np.abs(dfft2)) if vmax is None else vmax

            im = axs[pi, bi].imshow(np.abs(dfft2), interpolation='none', aspect='auto', cmap='turbo', 
                                    norm=matplotlib.colors.LogNorm(vmin=vmin, vmax=vmax),
                                    extent=[delays[0], delays[-1], frates[-1], frates[0]])
            axs[pi, bi].set_xlim([-799, 799])
            axs[pi, bi].set_ylim([-5, 5])

            # Get baseline-lengths
            blvec = antpos[bl[0]] - antpos[bl[1]]
            bllen = np.linalg.norm(blvec)
            if bi < 3:
                bltype = f"{int(np.round(bllen / 14.6))}-unit EW" 
            else:
                bltype = f"{np.abs(blvec[0]):.0f} m {'E' if blvec[0] > 0 else 'W'}, {np.abs(blvec[1]):.0f} m {'N' if blvec[1] > 0 else 'S'}"

            axs[pi, bi].text(0.04, 0.96, f"{bltype} ({pol})\n{np.round(data.freqs[rng[0]] / 1e6, 1)}—{np.round(data.freqs[rng[-1]] / 1e6, 1)} MHz", 
                             horizontalalignment='left', verticalalignment='top', 
                             color='black', transform=axs[pi, bi].transAxes, fontsize=12, 
                             bbox=dict(facecolor='w', alpha=.7, boxstyle='round'))

            # Label axes
            if pi == 1:
                axs[pi, bi].set_xlabel('Delay (ns)')
            if bi == 0:
                axs[pi, bi].set_ylabel('Fringe Rate [mHz]')
                
            if delay_bounds:
                axs[pi, bi].axvline(delay_bounds[bl[:2]][0] * 1e9, color='white', ls='-')
                axs[pi, bi].axvline(delay_bounds[bl[:2]][1] * 1e9, color='white', ls='-')
            if fringe_bounds:
                axs[pi, bi].axhline(fringe_bounds[bl[:2]][0], color='white', ls='-.')
                axs[pi, bi].axhline(fringe_bounds[bl[:2]][1], color='white', ls='-.')

    plt.tight_layout()

    # Colorbar
    fig.subplots_adjust(right=0.9)
    cbar_ax = fig.add_axes([0.92, 0.04, 0.02, 0.95])
    fig.colorbar(im, cax=cbar_ax, label=r'$|\widetilde{V}|\,\mathrm{[Jy]}$', extend='both')
    
    return {'vmin': vmin, 'vmax': vmax}

In [ ]:
plot_delay_vs_lst(
    data_inpaint, low_rng,
    title=f'Figure 2: Real Part of Redundantly-Averaged, Inpainted, Delay-Transformed Visibilities, {np.round(data.freqs[low_rng[0]] / 1e6, 1)}—{np.round(data.freqs[low_rng[-1]] / 1e6, 1)} MHz'
);

In [ ]:
plot_delay_vs_lst(
    data_inpaint, high_rng,
    title=f'Figure 3: Real Part of Redundantly-Averaged, Inpainted, Delay-Transformed Visibilities, {np.round(data.freqs[high_rng[0]] / 1e6, 1)}—{np.round(data.freqs[high_rng[-1]] / 1e6, 1)} MHz'
);

In [ ]:
low_band_meta = plot_delay_fringe_rate(
    data_inpaint, low_rng,
    title=f'Figure 4: Redundantly-Averaged, Inpainted, Delay + Fringe Transformed Visibilities, {np.round(data.freqs[low_rng[0]] / 1e6, 1)}—{np.round(data.freqs[low_rng[-1]] / 1e6, 1)} MHz'
)

In [ ]:
high_band_meta = plot_delay_fringe_rate(
    data_inpaint, high_rng,
    title=f'Figure 5: Redundantly-Averaged, Inpainted, Delay + Fringe Transformed Visibilities, {np.round(data.freqs[high_rng[0]] / 1e6, 1)}—{np.round(data.freqs[high_rng[-1]] / 1e6, 1)} MHz'
)

# *Section 2: Delay-Filter and Cross-Talk Filter Redundantly-Averaged Visibilities*

The delay filter subtracts a DPSS fit to each integration's unflagged channels out to the horizon plus `STANDOFF` (and at least
`FILT_MIN_DLY`), leaving zeros where the data are flagged. The cross-talk filter then subtracts each channel's DPSS fit in
time within `XTALK_FRATE` of zero fringe rate, as the pI SNR stage does. Below and above FM:
* Figures 6 and 7: the real part of the filtered delay spectra, against LST;
* Figures 8 and 9: their delay and fringe-rate transforms, with the delay filter's edges marked;
* Figures 10 and 11: delay spectra averaged over the night (coherently over 20 integrations, then incoherently), at each stage so far.

In [ ]:
data_xtalk_filt = deepcopy(data_delay_filt)

cache = {}
for bl in data_delay_filt:
    # don't bother delay filtering all 0s.
    if np.all(data_delay_filt[bl] == 0.0) and np.all(flags[bl]):
        continue

    # run delay filter for each band individually
    d_res = np.zeros_like(data_delay_filt[bl])

    # Get weights
    wgts = (~flags[bl]).astype(float)

    for band in [low_band, high_band]:
        _, d_res[:, band], info = dspec.fourier_filter(data.times * 24 * 60 * 60, data_delay_filt[bl][:, band],
                                                       wgts=wgts[:, band], filter_centers=[0],
                                                       filter_half_widths=[XTALK_FRATE / 1000], mode='dpss_solve',
                                                       eigenval_cutoff=[EIGENVAL_CUTOFF], max_contiguous_edge_flags=len(data.times), cache=cache, filter_dims=0)

    data_xtalk_filt[bl] = d_res

In [ ]:
_ = plot_delay_vs_lst(
    data_xtalk_filt, low_rng,
    title=f'Figure 6: Real Part of Redundantly-Averaged, DPSS Delay + Cross-Talk Filtered, Delay-Transformed Visibilities, {np.round(data.freqs[low_rng[0]] / 1e6, 1)}—{np.round(data.freqs[low_rng[-1]] / 1e6, 1)} MHz'
)

In [ ]:
_ = plot_delay_vs_lst(
    data_xtalk_filt, high_rng,
    title=f'Figure 7: Real Part of Redundantly-Averaged, DPSS Delay + Cross-Talk Filtered, Delay-Transformed Visibilities, {np.round(data.freqs[high_rng[0]] / 1e6, 1)}—{np.round(data.freqs[high_rng[-1]] / 1e6, 1)} MHz'
)

In [ ]:
low_band_meta = plot_delay_fringe_rate(
    data_xtalk_filt, low_rng, delay_bounds=delay_bounds,
    title=f'Figure 8: Redundantly-Averaged, DPSS Delay + Cross-Talk Filtered, Delay + Fringe Transformed Visibilities, {np.round(data.freqs[low_rng[0]] / 1e6, 1)}—{np.round(data.freqs[low_rng[-1]] / 1e6, 1)} MHz'
)

In [ ]:
high_band_meta = plot_delay_fringe_rate(
    data_xtalk_filt, high_rng, delay_bounds=delay_bounds,
    title=f'Figure 9: Redundantly-Averaged, DPSS Delay + Cross-Talk Filtered, Delay + Fringe Transformed Visibilities, {np.round(data.freqs[high_rng[0]] / 1e6, 1)}—{np.round(data.freqs[high_rng[-1]] / 1e6, 1)} MHz'
)

In [ ]:
def calculate_avg_norm_spectra(dfft, flags, nsamples):
    '''Calculates incoherently-averaged spectrum, using ses flags and frequency-averaged nsamples as weights.'''
    wgts = np.where(np.all(flags, axis=1, keepdims=True), np.nan, np.sum(nsamples, axis=1, keepdims=True))
    avg_norm_spectra = np.nanmean(np.where(np.all(flags, axis=1, keepdims=True), np.nan, np.abs(dfft) * wgts), axis=0) / np.nanmean(wgts)
    return avg_norm_spectra

def coherently_average_visibilities(dc_list, flags, nsamples, rng, nint=20):
    '''Averages each baseline's visibilities coherently over nint integrations (rephased to a common LST), leaving out the
    integrations it has flagged at every frequency, then averages their delay spectra incoherently over the night.'''
    avg_vis = []
    delays = uvtools.utils.fourier_freqs(dc_list[0].freqs[rng]) * 1e9

    for dc in dc_list:
        avg_delay_spec = {}
        for bl in dc.keys():
            bl_vec = (antpos[bl[1]] - antpos[bl[0]])
            flags_for_avg = np.repeat(np.all(flags[bl], axis=1, keepdims=True), dc.freqs.shape[0], axis=1)

            # Coherently average
            ad, af, an, _, _ = frf.timeavg_waterfall(
                dc[bl], nint, nsamples=nsamples[bl], flags=flags_for_avg,
                rephase=True, lsts=dc.lsts, freqs=dc.freqs, bl_vec=bl_vec,
                extra_arrays=dict(times=dc.times), wgt_by_nsample=True, verbose=False
            )

            # Delay Transform
            dfft_avg = uvtools.utils.FFT(ad[:, rng], axis=1, taper='bh7')

            # Incoherently Average - Cohorently Averaged Visibilities
            avg_delay_spec[bl] = calculate_avg_norm_spectra(
                dfft_avg, af[:, rng], an[:, rng]
            )

        avg_vis.append(avg_delay_spec)

    return delays, avg_vis

def plot_coherently_averaged_1d(delays, dc_list, plot_labels, rng, min_dly=FILT_MIN_DLY, ymin=3e-1, ymax=3e4, title=""):
    # Add search-able title to figure
    display(HTML(f'<h2>{title}</h2>'))
    
    fig, axs = plt.subplots(2, 4, figsize=(14, 10), sharex=True, sharey=True, gridspec_kw={'wspace': 0, 'hspace': 0})

    plot_bls = list(set([bl[:2] for bl in dc_list[0].keys() if bl[0] != bl[1]]))
    plot_bls = sorted(plot_bls, key=lambda k: np.linalg.norm(antpos[k[1]] - antpos[k[0]]))
    plot_pols = sorted(list(set([bl[-1] for bl in dc_list[0].keys()])))
    
    # Lists for setting vmin and vmax
    ymins = []
    ymaxs = []
        
    for bi, blkey in enumerate(plot_bls):
        for pi, pol in enumerate(plot_pols):
            
            # Store plot output for legend
            plot_objs = []
            
            bl = blkey + (pol, )
            for fi, avg_filt in enumerate(dc_list):
                line = axs[pi, bi].semilogy(delays, avg_filt[bl], label=plot_labels[fi])
                plot_objs.append(line[0])
                
                # Get vmins and maxes
                ymins.append(np.median(avg_filt[bl]))
                ymaxs.append(np.max(avg_filt[bl]))

            axs[pi, bi].set_xlim([-2900, 2900])
            axs[pi, bi].grid()

            # Set widths
            bl_vec = (antpos[bl[1]] - antpos[bl[0]])
            bl_len = np.linalg.norm(bl_vec[:2]) / vis_clean.constants.c.value
            filter_centers, filter_half_widths = vis_clean.gen_filter_properties(ax='freq', horizon=1, standoff=STANDOFF, 
                                                                                 min_dly=min_dly, bl_len=bl_len)
            axs[pi, bi].axvline(-filter_half_widths[0] * 1e9, color='k', ls=':')
            fw_line = axs[pi, bi].axvline(filter_half_widths[0] * 1e9, color='k', ls=':', label='Filter Width')
            plot_objs.append(fw_line)

            axs[pi, bi].axvline(-bl_len * 1e9, color='k', ls='-', lw=0.75)
            horizon_line = axs[pi, bi].axvline(bl_len * 1e9, color='k', ls='-', label='Horizon', lw=0.75)
            plot_objs.append(horizon_line)

            # Get baseline-lengths
            blvec = data.antpos[bl[0]] - data.antpos[bl[1]]
            bllen = np.linalg.norm(blvec)
            if bi < 3:
                bltype = f"{int(np.round(bllen / 14.6))}-unit EW" 
            else:
                bltype = f"{np.abs(blvec[0]):.0f} m {'E' if blvec[0] > 0 else 'W'}, {np.abs(blvec[1]):.0f} m {'N' if blvec[1] > 0 else 'S'}"

            axs[pi, bi].text(0.05, 0.97, f"{bltype} ({pol})\n{np.round(data.freqs[rng[0]] / 1e6, 1)}—{np.round(data.freqs[rng[-1]] / 1e6, 1)} MHz", 
                             horizontalalignment='left', verticalalignment='top', 
                             color='black', transform=axs[pi, bi].transAxes, fontsize=12, 
                             bbox=dict(facecolor='w', alpha=.7, boxstyle='round'))

            if bi == 0:
                axs[pi, bi].set_ylabel(r'$|\widetilde{V}|\,\mathrm{[Jy]}$')
            if pi == 1:
                axs[pi, bi].set_xlabel(r'$\tau \ [\rm ns]$')
                
    
    for bi, blkey in enumerate(plot_bls):
        for pi, pol in enumerate(plot_pols):
            axs[pi, bi].set_ylim([np.min(ymins) * 0.1, np.max(ymaxs) * 10])

    fig.legend([pobj for pobj in plot_objs], [pobj._label for pobj in plot_objs], loc='upper center', ncol=len(plot_objs), 
               bbox_to_anchor=(0.5, 1.025))
    plt.tight_layout()
    plt.show()

In [ ]:
# Coherently-average visibilities
delays, avg_vis = coherently_average_visibilities(
    [data_inpaint, data_delay_filt, data_xtalk_filt], flags, nsamples, low_rng
)

# Plot coherently-averaged visibilities
plot_labels = ['Inpainted', 'DPSS Delay-Filtered', f'DPSS Delay-Filtered + DPSS Notch-Filtered: {XTALK_FRATE} mHz']
plot_coherently_averaged_1d(
    delays, avg_vis, plot_labels, low_rng,
    title=f'Figure 10: Coherently-Averaged, Delay-Transformed, Incoherently-Averaged Visibilities, {np.round(data.freqs[low_rng[0]] / 1e6, 1)}—{np.round(data.freqs[low_rng[-1]] / 1e6, 1)} MHz'
)

In [ ]:
# Coherently-average visibilities
delays, avg_vis = coherently_average_visibilities(
    [data_inpaint, data_delay_filt, data_xtalk_filt], flags, nsamples, high_rng
)

# Plot coherently-averaged visibilities
plot_labels = ['Inpainted', 'DPSS Delay-Filtered', f'DPSS Delay-Filtered + DPSS Notch-Filtered: {XTALK_FRATE} mHz']
plot_coherently_averaged_1d(
    delays, avg_vis, plot_labels, high_rng,
    title=f'Figure 11: Coherently-Averaged, Delay-Transformed, Incoherently-Averaged Visibilities, {np.round(data.freqs[high_rng[0]] / 1e6, 1)}—{np.round(data.freqs[high_rng[-1]] / 1e6, 1)} MHz'
)

# *Section 3: Form Pseudo-Stokes I + Q*

Pseudo-Stokes I is (ee + nn) / 2 and Q is (ee - nn) / 2, at each stage. Below and above FM:
* Figures 12 and 13: the real part of the delay and cross-talk filtered delay spectra, against LST;
* Figures 14 and 15: their delay and fringe-rate transforms;
* Figures 16 and 17: delay spectra averaged over the night, at each stage.

In [ ]:
# Create dictionaries for polarized data
data_p_xtalk_filt, flags_p_xtalk_filt, nsamples_p_xtalk_filt = {}, {}, {}
data_p_delay_filt, flags_p_delay_filt, nsamples_p_delay_filt = {}, {}, {}
data_p_inpaint, flags_p_inpaint, nsamples_p_inpaint = {}, {}, {}

for bl in data_inpaint.antpairs():
    # Stokes-I - data, flags, namples
    data_p_xtalk_filt[bl + ('pI',)] = (data_xtalk_filt[bl + ('nn',)] + data_xtalk_filt[bl + ('ee',)]) / 2
    data_p_delay_filt[bl + ('pI',)] = (data_delay_filt[bl + ('nn',)] + data_delay_filt[bl + ('ee',)]) / 2
    data_p_inpaint[bl + ('pI',)] = (data_inpaint[bl + ('nn',)] + data_inpaint[bl + ('ee',)]) / 2
    
    flags_p_xtalk_filt[bl + ('pI',)] = flags[bl + ('nn',)] | flags[bl + ('ee',)]
    flags_p_delay_filt[bl + ('pI',)] = flags[bl + ('nn',)] | flags[bl + ('ee',)]
    flags_p_inpaint[bl + ('pI',)] = flags[bl + ('nn',)] | flags[bl + ('ee',)]
    
    nsamples_p_xtalk_filt[bl + ('pI',)] = (nsamples[bl + ('nn',)] + nsamples[bl + ('ee',)]) / 2
    nsamples_p_delay_filt[bl + ('pI',)] = (nsamples[bl + ('nn',)] + nsamples[bl + ('ee',)]) / 2
    nsamples_p_inpaint[bl + ('pI',)] = (nsamples[bl + ('nn',)] + nsamples[bl + ('ee',)]) / 2
    
    # Stokes Q - data, flags, namples
    data_p_xtalk_filt[bl + ('pQ',)] = (data_xtalk_filt[bl + ('ee',)] - data_xtalk_filt[bl + ('nn',)]) / 2
    data_p_delay_filt[bl + ('pQ',)] = (data_delay_filt[bl + ('ee',)] - data_delay_filt[bl + ('nn',)]) / 2
    data_p_inpaint[bl + ('pQ',)] = (data_inpaint[bl + ('ee',)] - data_inpaint[bl + ('nn',)]) / 2
    
    flags_p_xtalk_filt[bl + ('pQ',)] = flags[bl + ('nn',)] | flags[bl + ('ee',)]
    flags_p_delay_filt[bl + ('pQ',)] = flags[bl + ('nn',)] | flags[bl + ('ee',)]
    flags_p_inpaint[bl + ('pQ',)] = flags[bl + ('nn',)] | flags[bl + ('ee',)]
    
    nsamples_p_xtalk_filt[bl + ('pQ',)] = (nsamples[bl + ('nn',)] + nsamples[bl + ('ee',)]) / 2
    nsamples_p_delay_filt[bl + ('pQ',)] = (nsamples[bl + ('nn',)] + nsamples[bl + ('ee',)]) / 2
    nsamples_p_inpaint[bl + ('pQ',)] = (nsamples[bl + ('nn',)] + nsamples[bl + ('ee',)]) / 2
    
# Attach metadata
data_p_xtalk_filt = DataContainer(data_p_xtalk_filt)
data_p_xtalk_filt.times = data.times
data_p_xtalk_filt.freqs = data.freqs
data_p_xtalk_filt.antpos = data.antpos
data_p_xtalk_filt.lsts = data.lsts

data_p_delay_filt = DataContainer(data_p_delay_filt)
data_p_delay_filt.times = data.times
data_p_delay_filt.freqs = data.freqs
data_p_delay_filt.antpos = data.antpos
data_p_delay_filt.lsts = data.lsts

data_p_inpaint = DataContainer(data_p_inpaint)
data_p_inpaint.times = data.times
data_p_inpaint.freqs = data.freqs
data_p_inpaint.antpos = data.antpos
data_p_inpaint.lsts = data.lsts

In [ ]:
_ = plot_delay_vs_lst(
    data_p_xtalk_filt, low_rng,
    title=f'Figure 12: Real Part of Redundantly-Averaged, DPSS Delay + Cross-Talk Filtered, Delay-Transformed, Pseudo-Stokes I and Q Visibilities, {np.round(data.freqs[low_rng[0]] / 1e6, 1)}—{np.round(data.freqs[low_rng[-1]] / 1e6, 1)} MHz'
)

In [ ]:
_ = plot_delay_vs_lst(
    data_p_xtalk_filt, high_rng,
    title=f'Figure 13: Real Part of Redundantly-Averaged, DPSS Delay + Cross-Talk Filtered, Delay-Transformed, Pseudo-Stokes I and Q Visibilities, {np.round(data.freqs[high_rng[0]] / 1e6, 1)}—{np.round(data.freqs[high_rng[-1]] / 1e6, 1)} MHz'
)

In [ ]:
_ = plot_delay_fringe_rate(
    data_p_xtalk_filt, low_rng, delay_bounds=delay_bounds,
    title=f'Figure 14: Redundantly-Averaged, DPSS Delay + Cross-Talk Filtered, Delay + Fringe Transformed, Pseudo-Stokes I and Q Visibilities, {np.round(data.freqs[low_rng[0]] / 1e6, 1)}—{np.round(data.freqs[low_rng[-1]] / 1e6, 1)} MHz',
    **low_band_meta
)

In [ ]:
_ = plot_delay_fringe_rate(
    data_p_xtalk_filt, high_rng, delay_bounds=delay_bounds,
    title=f'Figure 15: Redundantly-Averaged, DPSS Delay + Cross-Talk Filtered, Delay + Fringe Transformed, Pseudo-Stokes I and Q Visibilities, {np.round(data.freqs[high_rng[0]] / 1e6, 1)}—{np.round(data.freqs[high_rng[-1]] / 1e6, 1)} MHz',
    **high_band_meta
)

In [ ]:
# Coherently-average visibilities
delays, avg_vis = coherently_average_visibilities(
    [data_p_inpaint, data_p_delay_filt, data_p_xtalk_filt], flags_p_inpaint, nsamples_p_inpaint, low_rng
)

# Plot coherently-averaged visibilities
plot_labels = [
    'Inpainted', 'DPSS Delay-Filtered', f'DPSS Delay-Filtered + DPSS Notch-Filtered: {XTALK_FRATE} mHz'
]
plot_coherently_averaged_1d(
    delays, avg_vis, plot_labels, low_rng,
    title=f'Figure 16: Coherently-Averaged, Delay-Transformed, Incoherently-Averaged Pseudo-Stokes I and Q Visibilities, {np.round(data.freqs[low_rng[0]] / 1e6, 1)}—{np.round(data.freqs[low_rng[-1]] / 1e6, 1)} MHz'
)

In [ ]:
# Coherently-average visibilities
delays, avg_vis = coherently_average_visibilities(
    [data_p_inpaint, data_p_delay_filt, data_p_xtalk_filt], flags_p_inpaint, nsamples_p_inpaint, high_rng
)

# Plot coherently-averaged visibilities
plot_labels = [
    'Inpainted', 'DPSS Delay-Filtered', f'DPSS Delay-Filtered + DPSS Notch-Filtered: {XTALK_FRATE} mHz'
]
plot_coherently_averaged_1d(
    delays, avg_vis, plot_labels, high_rng,
    title=f'Figure 17: Coherently-Averaged, Delay-Transformed, Incoherently-Averaged Pseudo-Stokes I and Q Visibilities, {np.round(data.freqs[high_rng[0]] / 1e6, 1)}—{np.round(data.freqs[high_rng[-1]] / 1e6, 1)} MHz'
)

# *Section 4: Main-Beam Fringe-Rate Filter*

Each baseline's main-beam fringe rates, between the 5th and 95th percentiles of the beam's fringe-rate response (from the beam
model in `UVBEAM_FILE`, for the nn polarization), are kept by a DPSS fit in time of the delay and cross-talk filtered
pseudo-Stokes I and Q. Below and above FM:
* Figures 18 and 19: the real part of the filtered delay spectra, against LST;
* Figures 20 and 21: their delay and fringe-rate transforms, with the main beam's fringe rates marked;
* Figures 22 and 23: delay spectra averaged over the night, at every stage.

In [ ]:
# Load uvbeam file
uvb = UVBeam()
uvb.read(UVBEAM_FILE)

# Extract metadata
dt = np.diff(data.times)[0] * 60 * 60 * 24
nfr = data.times.shape[0]
dfr = np.diff(uvtools.utils.fourier_freqs(dt * np.arange(nfr)))[0] * 1000

# Get fringe rates centers and widths for each baseline
frate_centers, frate_half_widths = frf.get_fringe_rate_limits(uvd=hd, uvb=uvb, nfr=nfr, dfr=dfr)
frate_bounds = {
    k[:2]: (frate_centers[k] - frate_half_widths[k], frate_centers[k] + frate_half_widths[k]) for k in frate_centers
}

# assign fringe rates centers and widths and bounds to each redundant key
for key in list(frate_centers.keys()):
    for bl in data.get_red(key):
        frate_centers[bl] = frate_centers[key]
        frate_half_widths[bl] = frate_half_widths[key]
        frate_bounds[bl[:2]] = frate_bounds[key[:2]]

In [ ]:
data_beam_filt = deepcopy(data_p_xtalk_filt)
cache = {}
for bl in data_p_xtalk_filt:
    # don't bother delay filtering all 0s.
    if np.all(data_p_xtalk_filt[bl] == 0.0) and np.all(flags_p_inpaint[bl]):
        continue

    d_mdl = np.zeros_like(data_p_xtalk_filt[bl])

    # Get weights
    wgts = (~np.repeat(np.all(flags_p_inpaint[bl], axis=1, keepdims=True), data.freqs.shape[0], axis=1)).astype(float)

    # Run main-beam filter for each band individually
    for band in [low_band, high_band]:
        d_mdl[:, band], _, info = dspec.fourier_filter(data.times * 24 * 60 * 60, data_p_xtalk_filt[bl][:, band], 
                                                       wgts=wgts[:, band], filter_centers=[frate_centers[bl[:2] + ('nn',)] / 1000], 
                                                       filter_half_widths=[frate_half_widths[bl[:2] + ('nn',)] / 1000], mode='dpss_solve',
                                                       eigenval_cutoff=[EIGENVAL_CUTOFF], max_contiguous_edge_flags=len(data.times), 
                                                       cache=cache, filter_dims=0)

    data_beam_filt[bl] = d_mdl

In [ ]:
_ = plot_delay_vs_lst(
    data_beam_filt, low_rng,
    title=f'Figure 18: Real Part of Redundantly-Averaged, DPSS Delay + Main-Beam Filtered, Delay-Transformed, Pseudo-Stokes I and Q Visibilities, {np.round(data.freqs[low_rng[0]] / 1e6, 1)}—{np.round(data.freqs[low_rng[-1]] / 1e6, 1)} MHz'
)

In [ ]:
_ = plot_delay_vs_lst(
    data_beam_filt, high_rng,
    title=f'Figure 19: Real Part of Redundantly-Averaged, DPSS Delay + Main-Beam Filtered, Delay-Transformed, Pseudo-Stokes I and Q Visibilities, {np.round(data.freqs[high_rng[0]] / 1e6, 1)}—{np.round(data.freqs[high_rng[-1]] / 1e6, 1)} MHz'
)

In [ ]:
_ = plot_delay_fringe_rate(
    data_beam_filt, low_rng, delay_bounds=delay_bounds, fringe_bounds=frate_bounds,
    title=f'Figure 20: Redundantly-Averaged, DPSS Delay + Cross-Talk + Main-Beam Filtered, Delay + Fringe Transformed, Pseudo-Stokes I and Q Visibilities, {np.round(data.freqs[low_rng[0]] / 1e6, 1)}—{np.round(data.freqs[low_rng[-1]] / 1e6, 1)} MHz',
    **low_band_meta
)

In [ ]:
_ = plot_delay_fringe_rate(
    data_beam_filt, high_rng, delay_bounds=delay_bounds, fringe_bounds=frate_bounds,
    title=f'Figure 21: Redundantly-Averaged, DPSS Delay + Cross-Talk + Main-Beam Filtered, Delay + Fringe Transformed, Pseudo-Stokes I and Q Visibilities, {np.round(data.freqs[high_rng[0]] / 1e6, 1)}—{np.round(data.freqs[high_rng[-1]] / 1e6, 1)} MHz',
    **high_band_meta
)

In [ ]:
# Coherently-average visibilities
delays, avg_vis = coherently_average_visibilities(
    [data_p_inpaint, data_p_delay_filt, data_p_xtalk_filt, data_beam_filt], flags_p_inpaint, nsamples_p_inpaint, low_rng
)

plot_labels = [
    'Inpainted', 'DPSS Delay-Filtered', 'DPSS Delay + Cross Talk-Filtered',
    f'DPSS Delay-Filtered + Main-Beam Filter'
]
plot_coherently_averaged_1d(
    delays, avg_vis, plot_labels, low_rng,
    title=f'Figure 22: Coherently-Averaged, Delay-Transformed, Incoherently-Averaged, Main-Beam Filtered Pseudo-Stokes I and Q Visibilities, {np.round(data.freqs[low_rng[0]] / 1e6, 1)}—{np.round(data.freqs[low_rng[-1]] / 1e6, 1)} MHz'
)

In [ ]:
# Coherently-average visibilities
delays, avg_vis = coherently_average_visibilities(
    [data_p_inpaint, data_p_delay_filt, data_p_xtalk_filt, data_beam_filt], flags_p_inpaint, nsamples_p_inpaint, high_rng
)

plot_labels = [
    'Inpainted', 'DPSS Delay-Filtered', 'DPSS Delay + Cross Talk-Filtered',
    f'DPSS Delay-Filtered + Main-Beam Filter'
]
plot_coherently_averaged_1d(
    delays, avg_vis, plot_labels, high_rng,
    title=f'Figure 23: Coherently-Averaged, Delay-Transformed, Incoherently-Averaged, Main-Beam Filtered Pseudo-Stokes I and Q Visibilities, {np.round(data.freqs[high_rng[0]] / 1e6, 1)}—{np.round(data.freqs[high_rng[-1]] / 1e6, 1)} MHz'
)

## Metadata

In [ ]:
for repo in ['numpy', 'scipy', 'astropy', 'hera_cal', 'hera_filters', 'uvtools', 'hera_notebook_templates', 'pyuvdata']:
    exec(f'from {repo} import __version__')
    print(f'{repo}: {__version__}')

print(f'Finished execution in {(time.time() - tstart) / 60:.2f} minutes.')